In [3]:

import pandas as pd
import electricity
import util

import plotly.express as px
import plotly.graph_objects as go

import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
utility = electricity.get_utility()

df = pd.json_normalize(utility)
df.head()

,Utility.Number,Utility.Name,Utility.State,Utility.Type,Demand.Summer Peak,Demand.Winter Peak,Sources.Generation,Sources.Purchased,Sources.Other,Sources.Total,...,Retail.Commercial.Customers,Retail.Industrial.Revenue,Retail.Industrial.Sales,Retail.Industrial.Customers,Retail.Transportation.Revenue,Retail.Transportation.Sales,Retail.Transportation.Customers,Retail.Total.Revenue,Retail.Total.Sales,Retail.Total.Customers
0,34,City of Abbeville - (SC),SC,Municipal,13.7,10.8,7000.0,59000.0,0.0,66000.0,...,460.0,0.0,0.0,0.0,0.0,0.0,0.0,7536.0,58000.0,3844.0
1,55,City of Aberdeen - (MS),MS,Municipal,32.4,30.3,0.0,209454.0,0.0,209454.0,...,662.0,5638.0,120537.0,1.0,0.0,0.0,0.0,14797.0,204261.0,3229.0
2,59,City of Abbeville - (LA),LA,Municipal,28.9,22.0,0.0,137264.0,0.0,137264.0,...,887.0,3011.1,35881.0,27.0,0.0,0.0,0.0,12383.0,127579.0,5494.0
3,84,A & N Electric Coop,VA,Cooperative,154.0,162.4,596.0,743457.0,0.0,744053.0,...,4227.0,15516.0,176162.0,8.0,0.0,0.0,0.0,78507.0,704010.0,35934.0
4,87,City of Ada - (MN),MN,Municipal,2.1,2.2,0.0,20028.0,0.0,20028.0,...,255.0,190.0,2615.0,58.0,0.0,0.0,0.0,1593.0,20028.0,1185.0


In [5]:
# df[["Uses.Retail", "Uses.Resale", "Uses.No Charge", "Uses.Consumed", "Uses.Losses", "Uses.Total"]]

In [6]:
ny_df = util.get_state_data('NY', df)
ny_df = util.prepare_data(ny_df)

# ny_df.info()

KeyError: 'RetailRevenue'

In [ ]:
# ny_df[ny_df.IndustrialPriceRatio.isna()]

In [ ]:
def plot_scatterplot(df):
    fig = px.scatter(ny_df, x='LoadFactor', y='ResidentialRetailRate', color="Utility.Type",
                    title="",
                    labels={
                            "LoadFactor": "System Efficiency (Load Factor)",
                            "ResidentialRetailRate": "Residential Rate ($/MWh)"
                        },
                    hover_name="Utility.Name")
    fig.show()

plot_scatterplot(ny_df)

In [ ]:
def plot_dumbbell(df):
    # Sort by spread to show the most "unfair" utilities at the top
    df_sorted = df[df.PriceSpread != 0].sort_values('PriceSpread', ascending=True).tail(10)

    fig = go.Figure()

    # 1. Add the lines connecting the dots
    for i, row in df_sorted.iterrows():
        fig.add_shape(
            type='line', x0=row['IndustrialRetailRate'], x1=row['ResidentialRetailRate'],
            y0=row['Utility.Name'], y1=row['Utility.Name'],
            line=dict(color='lightgrey', width=2)
        )

    # 2. Add Industrial dots
    fig.add_trace(go.Scatter(
        x=df_sorted['IndustrialRetailRate'], y=df_sorted['Utility.Name'],
        mode='markers', name='Industrial Rate', marker=dict(color='#1f77b4', size=10)
    ))

    # 3. Add Residential dots
    fig.add_trace(go.Scatter(
        x=df_sorted['ResidentialRetailRate'], y=df_sorted['Utility.Name'],
        mode='markers', name='Residential Rate', marker=dict(color='#d62728', size=10)
    ))

    fig.update_layout(title="Rate Disparity by Utility", xaxis_title="$/MWh", yaxis_title="")
    return fig

ps_df = ny_df[ny_df.IndustrialRetailRate != 0]
ps_df = ps_df[ps_df.ResidentialRetailRate != 0]

plot_dumbbell(ps_df)

In [ ]:
def plot_distributions(df):
    plt.figure(figsize=(10, 6))
    sns.set_style("whitegrid")
    
    # Compare Residential Price across Utility Types
    ax = sns.boxplot(
        data=df, 
        x='Utility.Type', 
        y='ResidentialRetailRate', 
        palette="Set2",
        hue="Utility.Type",
        showfliers=False # Removes extreme outliers for a cleaner "box" view
    )
    
    # Add a swarmplot on top to show individual data points
    sns.swarmplot(data=df, x='Utility.Type', y='ResidentialRetailRate', color=".25", size=4)
    
    plt.title("Residential Price Distribution by Utility Ownership Type")
    plt.ylabel("Residential Rate ($/MWh)")
    plt.xlabel("Ownership Model")
    
    plt.show()

plot_distributions(ny_df)

In [ ]:
def get_state_box_plot(df):
    fig = px.box(df, x='Utility.Type', y='ResidentialRetailRate', color="Utility.Type")
    fig.show()

get_state_box_plot(ny_df)

In [ ]:
def plot_sankey(row):
    # 'row' is a single utility's data
    labels = ["Generation", "Purchased", "Other Source", "Total Sources", "Retail Sales", "Losses", "Other Uses"]
    
    # Convert raw values to percentages of the 'Total Sources'
    total = row['Sources.Total']

    sources_gen_pct = (row['Sources.Generation'] / total) * 100
    sources_pur_pct = (row['Sources.Purchased'] / total) * 100
    sources_oth_pct = (row['Sources.Other'] / total) * 100
    uses_retail_pct = (row['Uses.Retail'] / total) * 100
    uses_loss_pct = (row['Uses.Losses'] / total) * 100

    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 15, thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = labels
        ),
        link = dict(
          source = [0, 1, 2, 3, 3, 3], # Indices of 'labels'
          target = [3, 3, 3, 4, 5, 6], 
          value = [
              sources_gen_pct, 
              sources_pur_pct, 
              sources_oth_pct,
              uses_retail_pct, 
              uses_loss_pct,
              max(0, 100 - (uses_retail_pct + uses_loss_pct))
          ]
      ))])

    fig.update_layout(title_text=f"Energy Flow %: {row['Utility.Name']}")
    return fig

plot_sankey(ny_df.iloc[12])